In [1]:
# import libraries for reading data
import pandas as pd
import matplotlib.pyplot as plt
import os
from PIL import Image
import cv2
import re
import torch
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from torchvision import transforms, models
import ast
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights
import torch.nn as nn
from sklearn.model_selection import train_test_split


<jemalloc>: Unsupported system page size


### Einlesen der Daten und Übersicht über die Daten

In [2]:
# data paths
train1_images_path = "/datasets/multi-view-pig-posture-recognition/train1_images"
train2_images_path = "/datasets/multi-view-pig-posture-recognition/train2_images"
test_images_path = "/datasets/multi-view-pig-posture-recognition/test_images"

# csv path with row_id, image_id, width, height, bbox, class_id
train1_csv_path = "/datasets/multi-view-pig-posture-recognition/train1.csv"
train2_csv_path = "/datasets/multi-view-pig-posture-recognition/train2.csv"
test_csv_path = "/datasets/multi-view-pig-posture-recognition/test.csv"

# txt file path
pig_posture_txt = "/datasets/multi-view-pig-posture-recognition/pig_posture_classes.txt"

In [3]:
# read all files and show statistics and content of csv files, column names etc.
# read csv files
train1_df = pd.read_csv(train1_csv_path)
train2_df = pd.read_csv(train2_csv_path)
test_df = pd.read_csv(test_csv_path)

# show column names of csv files
print("\nTrain1 CSV Columns:")
print(train1_df.columns)
print("\nTrain2 CSV Columns:")
print(train2_df.columns)
print("\nTest CSV Columns:")
print(test_df.columns)

# show content of txt file
with open(pig_posture_txt, 'r') as f:
    pig_posture_content = f.read()

# show numbers of unique image_ids, row_ids in train1, train2 and test csv files
print("\nNumber of unique image_ids in Train1 CSV:", train1_df['image_id'].nunique())
print("Number of unique image_ids in Train2 CSV:", train2_df['image_id'].nunique())
print("Number of unique row_ids in Train1 CSV:", train1_df['row_id'].nunique())
print("Number of unique row_ids in Train2 CSV:", train2_df['row_id'].nunique())
print("\nPig Posture Classes:")
print(pig_posture_content)




Train1 CSV Columns:
Index(['row_id', 'image_id', 'width', 'height', 'bbox', 'class_id'], dtype='object')

Train2 CSV Columns:
Index(['row_id', 'image_id', 'width', 'height', 'bbox', 'class_id'], dtype='object')

Test CSV Columns:
Index(['row_id', 'image_id', 'width', 'height', 'bbox'], dtype='object')

Number of unique image_ids in Train1 CSV: 3090
Number of unique image_ids in Train2 CSV: 3150
Number of unique row_ids in Train1 CSV: 22934
Number of unique row_ids in Train2 CSV: 23450

Pig Posture Classes:
Lateral_lying_left
Lateral_lying_right
Sitting
Standing
Sternal_lying



In [4]:
train1_df.head()

,row_id,image_id,width,height,bbox,class_id
0,train_pen1_orb_cam1_20250108_085204_0000,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[1031.5,368.0,349.0,435.0]",0
1,train_pen1_orb_cam1_20250108_085204_0001,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[1278.5,428.0,233.0,438.0]",4
2,train_pen1_orb_cam1_20250108_085204_0002,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[732.0,137.5,342.0,198.0]",1
3,train_pen1_orb_cam1_20250108_085204_0003,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[830.0,169.0,370.0,263.0]",0
4,train_pen1_orb_cam1_20250108_085204_0004,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[611.5,314.8,381.5,386.6]",3


In [5]:
train2_df.head()

,row_id,image_id,width,height,bbox,class_id
0,train_pen1_orb_cam1_20250108_085204_0000,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[1031.5,368.0,349.0,435.0]",0
1,train_pen1_orb_cam1_20250108_085204_0001,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[1278.5,428.0,233.0,438.0]",4
2,train_pen1_orb_cam1_20250108_085204_0002,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[732.0,137.5,342.0,198.0]",1
3,train_pen1_orb_cam1_20250108_085204_0003,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[830.0,169.0,370.0,263.0]",0
4,train_pen1_orb_cam1_20250108_085204_0004,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[611.5,314.8,381.5,386.6]",3


### EDA 
Class Definitions:

0 — Lateral_lying_left

1 — Lateral_lying_right

2 — Sitting

3 — Standing

4 — Sternal_lying

In [6]:
# show amount of class distribution in train1 and train2 csv files

train1_df_distribution = train1_df['class_id'].value_counts().sort_index()
train2_df_distribution = train2_df['class_id'].value_counts().sort_index()

print("\nClass Distribution in Train1 CSV:")
print(train1_df_distribution)
print("\nClass Distribution in Train2 CSV:")
print(train2_df_distribution)


Class Distribution in Train1 CSV:
class_id
0    3053
1    3376
2     680
3    9617
4    6208
Name: count, dtype: int64

Class Distribution in Train2 CSV:
class_id
0    3083
1    3435
2     695
3    9928
4    6309
Name: count, dtype: int64


### Klassen sind stark unausgewogen, insbesondere Sitting Class id = 2. Gegenmaßnahme ist notwendig, um die Minderheitsklasse nicht zu vernachlässigen.

In [7]:
# check if there are any missing values in train1 and train2 csv files
print("\nMissing values in Train1 CSV:")
print(train1_df.isnull().sum())
print("\nMissing values in Train2 CSV:")
print(train2_df.isnull().sum())


Missing values in Train1 CSV:
row_id      0
image_id    0
width       0
height      0
bbox        0
class_id    0
dtype: int64

Missing values in Train2 CSV:
row_id      0
image_id    0
width       0
height      0
bbox        0
class_id    0
dtype: int64


In [8]:
# check if unique values in "height" and "weight" columns in train1 and train2 csv files are the same
print("\nUnique values in 'height' column in Train1 CSV:")
print(train1_df['height'].unique())
print("\nUnique values in 'height' column in Train2 CSV:")
print(train2_df['height'].unique())
print("\nUnique values in 'width' column in Train1 CSV:")
print(train1_df['width'].unique())
print("\nUnique values in 'width' column in Train2 CSV:")
print(train2_df['width'].unique())


Unique values in 'height' column in Train1 CSV:
[1080  720 1520]

Unique values in 'height' column in Train2 CSV:
[1080  720 1520]

Unique values in 'width' column in Train1 CSV:
[1920 1280 2688]

Unique values in 'width' column in Train2 CSV:
[1920 1280 2688]


### Die Bilder liegen nur in drei Auflösungen vor: 1280 x 720, 1920 x 1080, 2688 x 1520. Vorverarbeitung ist konsistent planbar. 

In [9]:
# Blur-Score (OpenCV)
def blur_score(img_bgr):
    return cv2.Laplacian(img_bgr, cv2.CV_64F).var()

# Helligkeit in HSV
def mean_brightness(img_bgr):
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    return float(hsv[...,2].mean())


In [10]:
# plot blur score and mean brightness for 100 random images in separate scatter plots
def plot_blur_brightness(df, images_path, num_images=100):
    random_images = df['image_id'].drop_duplicates().sample(num_images).values
    blur_scores = []
    brightness_values = []
    
    for image_id in random_images:
        image_path = os.path.join(images_path, image_id)
        img_bgr = cv2.imread(image_path)
        
        blur_scores.append(blur_score(img_bgr))
        brightness_values.append(mean_brightness(img_bgr))
    
    # Scatter plot for Blur Score
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.scatter(range(num_images), blur_scores, color='blue')
    plt.title('Blur Score of Random Images')
    plt.xlabel('Image Index')
    plt.ylabel('Blur Score')
    
    # Scatter plot for Mean Brightness
    plt.subplot(1, 2, 2)
    plt.scatter(range(num_images), brightness_values, color='orange')
    plt.title('Mean Brightness of Random Images')
    plt.xlabel('Image Index')
    plt.ylabel('Mean Brightness')
    
    plt.tight_layout()
    plt.show()


### Fazit: Die EDA zeigt, dass die Klassen in den Trainingsdaten relativ ausgewogen verteilt sind, was für das Training eines Modells vorteilhaft ist. Es gibt keine fehlenden Werte in den CSV-Dateien, und die Bildgrößen sind konsistent. Die Analyse der Bildqualität anhand von Blur-Score und Helligkeit zeigt eine gewisse Variation. Im nächsten Schritt möchte ich die Kameras trennen und die Bilder entsprechend der Kamera analysieren, um mögliche Unterschiede in der Bildqualität oder den Aufnahmewinkeln zu identifizieren.

### Trennung der Kameras anhand der Bildnamen nur in Train1

In [11]:
def extract_pen_id(s: str):
    m = re.search(r'^(pen\d+)', s)
    return m.group(1) if m else None

def extract_camera_type(s: str):
    m = re.search(r'_(orb|tur)_', s)
    return m.group(1) if m else None

def extract_camera_number(s: str):
    m = re.search(r'cam(\d+)', s)
    return m.group(1) if m else None

# df1 = train1.csv als DataFrame; ersetze 'FILENAME_COL' durch deine Spalte (z. B. 'image', 'file_name', ...).
FILENAME_COL = "image_id"
train1_df["pen_id"]        = train1_df[FILENAME_COL].apply(extract_pen_id)
train1_df["camera_type"]   = train1_df[FILENAME_COL].apply(extract_camera_type)
train1_df["camera_number"] = train1_df[FILENAME_COL].apply(extract_camera_number)
train1_df["camera_view_id"]      = train1_df["pen_id"] + "_" + train1_df["camera_type"] + "_cam" + train1_df["camera_number"]


In [12]:
train1_df.head()

,row_id,image_id,width,height,bbox,class_id,pen_id,camera_type,camera_number,camera_view_id
0,train_pen1_orb_cam1_20250108_085204_0000,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[1031.5,368.0,349.0,435.0]",0,pen1,orb,1,pen1_orb_cam1
1,train_pen1_orb_cam1_20250108_085204_0001,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[1278.5,428.0,233.0,438.0]",4,pen1,orb,1,pen1_orb_cam1
2,train_pen1_orb_cam1_20250108_085204_0002,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[732.0,137.5,342.0,198.0]",1,pen1,orb,1,pen1_orb_cam1
3,train_pen1_orb_cam1_20250108_085204_0003,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[830.0,169.0,370.0,263.0]",0,pen1,orb,1,pen1_orb_cam1
4,train_pen1_orb_cam1_20250108_085204_0004,pen1_orb_cam1_20250108_085204.jpg,1920,1080,"[611.5,314.8,381.5,386.6]",3,pen1,orb,1,pen1_orb_cam1


In [13]:
# show unique values in camera_view_id column
print("\nUnique values in 'camera_view_id' column in Train1 CSV:")
print(train1_df['camera_view_id'].unique())



Unique values in 'camera_view_id' column in Train1 CSV:
['pen1_orb_cam1' 'pen1_orb_cam2' 'pen1_tur_cam2' 'pen2_orb_cam1'
 'pen2_tur_cam1']


In [14]:
# show amount of images per camera view from train1_df
train1_camera_view_counts = train1_df['camera_view_id'].value_counts()
train1_camera_view_counts = train1_camera_view_counts.reset_index()
train1_camera_view_counts.columns = ['camera_view_id', 'image_count']
train1_camera_view_counts = train1_camera_view_counts.sort_values(by='image_count', ascending=False)
train1_camera_view_counts

,camera_view_id,image_count
0,pen2_tur_cam1,9713
1,pen1_tur_cam2,8194
2,pen2_orb_cam1,2817
3,pen1_orb_cam2,1488
4,pen1_orb_cam1,722


### Trennung der Kameras anhand der Bildnamen nur in Train2

In [15]:
train2_df["pen_id"]        = train2_df["image_id"].apply(extract_pen_id)
train2_df["camera_type"]   = train2_df["image_id"].apply(extract_camera_type)
train2_df["camera_number"] = train2_df["image_id"].apply(extract_camera_number)
train2_df["camera_view_id"]      = train2_df["pen_id"] + "_" + train2_df["camera_type"] + "_cam" + train2_df["camera_number"]


In [16]:
# show unique values in camera_view_id column
print("\nUnique values in 'camera_view_id' column in Train2 CSV:")
print(train2_df['camera_view_id'].unique())



Unique values in 'camera_view_id' column in Train2 CSV:
['pen1_orb_cam1' 'pen1_orb_cam2' 'pen1_tur_cam1' 'pen1_tur_cam2'
 'pen2_orb_cam1' 'pen2_orb_cam2' 'pen2_tur_cam1' 'pen2_tur_cam2']


In [17]:
# show amount of images per camera view from train2_df
train2_camera_view_counts = train2_df['camera_view_id'].value_counts()
train2_camera_view_counts = train2_camera_view_counts.reset_index()
train2_camera_view_counts.columns = ['camera_view_id', 'image_count']
train2_camera_view_counts = train2_camera_view_counts.sort_values(by='image_count', ascending=False)
train2_camera_view_counts


,camera_view_id,image_count
0,pen2_tur_cam1,9713
1,pen1_tur_cam2,8194
2,pen2_orb_cam1,2817
3,pen1_orb_cam2,1488
4,pen1_orb_cam1,722
5,pen1_tur_cam1,200
6,pen2_tur_cam2,196
7,pen2_orb_cam2,120


### Erster Versuch eines Modelltrainings mit dem Modell "MobileNetv3". Vorbereitungen treffen mit transforms. 

In [18]:
# transforms for data augmentation and data preprocessing
img_size = (224, 224)
normalize_mean = [0.485, 0.456, 0.406]
normalize_std = [0.229, 0.224, 0.225]


train_transforms = transforms.Compose([
    transforms.Resize(img_size),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=normalize_mean, std=normalize_std),
])

val_transforms = transforms.Compose([
    transforms.Resize(img_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=normalize_mean, std=normalize_std),
])



In [19]:

class PigCropDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.cache = [None] * len(self.df)  # Speicherplatz im RAM reservieren
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Wenn im RAM vorhanden, nimm das (Bausatz-Prinzip)
        if self.cache[idx] is not None:
            crop_img, label = self.cache[idx]
        else:
            # Ansonsten: Einmalig den schweren Prozess durchlaufen (Laden & Schneiden)
            r = self.df.iloc[idx]
            p = os.path.join(self.image_dir, r["image_id"])
            image = Image.open(p).convert("RGB")
            
            bbox = ast.literal_eval(r["bbox"]) if isinstance(r["bbox"], str) else r["bbox"]
            x1, y1, w, h = bbox
            # PIL .crop ist schneller als numpy-slicing für diesen Zweck
            crop_img = image.crop((x1, y1, x1 + w, y1 + h))
            label = int(r["class_id"])
            
            # Im RAM speichern für die nächste Epoche
            self.cache[idx] = (crop_img, label)

        # Transformationen (Augmentation) immer erst NACH dem Cache anwenden,
        # damit jede Epoche ein leicht anderes Bild sieht (Training wird besser).
        if self.transform:
            return self.transform(crop_img), torch.tensor(label, dtype=torch.long)
        
        return crop_img, torch.tensor(label, dtype=torch.long)



In [20]:
sampleDS = PigCropDataset(train1_df, train1_images_path, transform=train_transforms)
loader = DataLoader(sampleDS, batch_size=9, shuffle=True)
for x, y in loader:
    print("Batch Image Shape:", x.shape)  # erwartet: [9, 3, 224, 224]
    print("Batch Label Shape:", y.shape)  # erwartet: [9]
    print("First Label:", y[0].item())    # 0..4
    break


Batch Image Shape: torch.Size([9, 3, 224, 224])
Batch Label Shape: torch.Size([9])
First Label: 3


In [21]:
class PigPostureCNN(nn.Module):
    def __init__(self, num_classes=5):
        super(PigPostureCNN, self).__init__()
        
        # Backbone: MobileNetV3 statt ResNet18
        weights = MobileNet_V3_Large_Weights.DEFAULT
        original_model = mobilenet_v3_large(weights=weights)
        
        # Features und Pooling extrahieren
        self.backbone = nn.Sequential(
            original_model.features,
            original_model.avgpool
        )
        
        # Backbone einfrieren (wie bei der Lehrkraft)
        for param in self.backbone.parameters():
            param.requires_grad = False
        
        # Label_Classifier (Struktur der Lehrkraft beibehalten)
        in_features = original_model.classifier[0].in_features
        self.label_classifier = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.backbone(x)
        x = torch.flatten(x, 1)
        label_preds = self.label_classifier(x)
        return label_preds

# Instanziierung auf cuda:3
device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
model = PigPostureCNN(num_classes=5).to(device)

In [22]:
class Learner:
    def __init__(self, model, train_dl, val_dl, device=None):
        self.model = model
        self.train_dl = train_dl
        self.val_dl = val_dl
        self.device = device
        
        self.model = self.model.to(self.device)
        self.loss_fn_classifier = nn.CrossEntropyLoss()
        self.best_acc = 0
        self.freeze()
        
    def freeze(self):
        for param in self.model.backbone.parameters():
            param.requires_grad = False
        
    def unfreeze(self):
        for param in self.model.backbone.parameters():
            param.requires_grad = True

    def fit(self, epochs, lr=1e-3):
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=lr)
        self.scheduler = torch.optim.lr_scheduler.OneCycleLR(
            self.optimizer, max_lr=lr*10, total_steps=epochs*len(self.train_dl)
        )
        
        for epoch in range(epochs):
            self.model.train()
            for xb, yb in tqdm(self.train_dl, desc=f"Epoch {epoch+1}"):
                xb, yb = xb.to(self.device), yb.to(self.device)
                
                self.optimizer.zero_grad()
                preds = self.model(xb)
                loss = self.loss_fn_classifier(preds, yb)
                loss.backward()
                self.optimizer.step()
                self.scheduler.step()
            
            # Validierung
            self.model.eval()
            correct = 0
            with torch.no_grad():
                for xb, yb in self.val_dl:
                    xb, yb = xb.to(self.device), yb.to(self.device)
                    preds = self.model(xb)
                    correct += (preds.argmax(1) == yb).sum().item()
            
            acc = correct / len(self.val_dl.dataset)
            if acc > self.best_acc: self.best_acc = acc
            print(f"Validation Accuracy: {acc:.4f}")

In [23]:
train_data, val_data = train_test_split(
    train1_df, 
    test_size=0.15, 
    random_state=42, 
    stratify=train1_df['class_id']
)

print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")

Training samples: 18347
Validation samples: 4587


In [26]:
# --- VORBEREITUNG ---
# Nutze hier die Version von PigCropDataset, in die wir das Caching (RAM-Speicher) 
# eingebaut haben, damit es ab Epoche 2 extrem schnell geht.
train_ds = PigCropDataset(train_data, train1_images_path, transform=train_transforms)
val_ds = PigCropDataset(val_data, train1_images_path, transform=val_transforms)

# DataLoaders - Optimierung: pin_memory auch für Validierung nutzen
batch_size = 128
train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=16, pin_memory=True)
val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=16, pin_memory=True)

# --- TRAINING STARTEN ---
# 1. Modell instanziieren
model = PigPostureCNN(num_classes=5).to(device)

# 2. Learner initialisieren
learner = Learner(model, train_dl, val_dl, device=device)

# 3. Sicherstellen, dass der Backbone eingefroren ist (WICHTIG für Phase 1)
learner.freeze()

# 4. Erste 10 Epochen trainieren (Classifier lernt die Grundlagen)
learner.fit(epochs=10, lr=1e-3)

Epoch 1: 100%|██████████| 144/144 [00:40<00:00,  3.56it/s]


Validation Accuracy: 0.7353


Epoch 2: 100%|██████████| 144/144 [00:40<00:00,  3.57it/s]


Validation Accuracy: 0.7576


Epoch 3: 100%|██████████| 144/144 [00:41<00:00,  3.44it/s]


...

Validation Accuracy: 0.8230


Epoch 9: 100%|██████████| 144/144 [00:40<00:00,  3.57it/s]


Validation Accuracy: 0.8295


Epoch 10: 100%|██████████| 144/144 [00:41<00:00,  3.47it/s]


Validation Accuracy: 0.8306


In [27]:
# --- PHASE 2: FINE-TUNING ---
# Jetzt tauen wir das Modell auf, um die Details für den Kaggle-Score zu lernen
learner.unfreeze()

# Wir trainieren weiter, aber mit einer VIEL kleineren Lernrate
learner.fit(epochs=20, lr=1e-5)

Epoch 1: 100%|██████████| 144/144 [00:42<00:00,  3.38it/s]


Validation Accuracy: 0.8363


Epoch 2: 100%|██████████| 144/144 [00:41<00:00,  3.47it/s]


Validation Accuracy: 0.8483


Epoch 3: 100%|██████████| 144/144 [00:39<00:00,  3.67it/s]


...

Validation Accuracy: 0.9392


Epoch 19: 100%|██████████| 144/144 [00:44<00:00,  3.23it/s]


Validation Accuracy: 0.9390


Epoch 20: 100%|██████████| 144/144 [00:39<00:00,  3.65it/s]


Validation Accuracy: 0.9394


### Erste Submission bei 10 Epochen hat eine Accuracy von 0.310 bei Kaggle erreicht. Erheblich von dem entfernt, was hier als Validation Accuracy von 0.83 zuletzt angezeigt wird